# 04 · Panel Construction

Joins DGStats, CAISO monthly, DeepSolar, and utility territory map into `panel_for_regression.parquet`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

PROC = ROOT / 'data' / 'processed'
EXT = ROOT / 'data' / 'external'

import pandas as pd
import geopandas as gpd
import numpy as np
from feature_engineering import build_panel

## 4.1 Build ZIP → Utility Territory Map

In [ ]:
utility_gdf = gpd.read_file(EXT / 'utility_territories.geojson')
print(utility_gdf.crs, utility_gdf.shape)
utility_gdf.head(2)

In [ ]:
# Read CA ZCTA shapefile — use Census TIGER ZCTA5
# If not yet downloaded, note the path and skip for now
zcta_path = EXT / 'ca_zcta.geojson'
if zcta_path.exists():
    zcta_gdf = gpd.read_file(zcta_path)
    zcta_gdf = zcta_gdf.to_crs(utility_gdf.crs)
    zcta_centroids = zcta_gdf.copy()
    zcta_centroids['geometry'] = zcta_gdf.geometry.centroid
    joined = gpd.sjoin(zcta_centroids[['ZCTA5CE20', 'geometry']], utility_gdf[['utility_name', 'geometry']], how='left', predicate='within')
    zip_utility = joined[['ZCTA5CE20', 'utility_name']].rename(columns={'ZCTA5CE20': 'zip_code', 'utility_name': 'utility'})
    zip_utility['zip_code'] = zip_utility['zip_code'].astype(str).str.zfill(5)
    zip_utility.to_parquet(PROC / 'zip_utility_map.parquet', index=False)
    print(f'ZIP → utility map: {len(zip_utility):,} ZIPs')
else:
    print(f'ZCTA shapefile not found at {zcta_path}. Using placeholder empty map.')
    zip_utility = pd.DataFrame(columns=['zip_code', 'utility'])

## 4.2 Merge into Regression Panel

In [ ]:
dgstats_panel = pd.read_parquet(PROC / 'dgstats_panel.parquet')
caiso_monthly = pd.read_parquet(PROC / 'caiso_monthly.parquet')
deepsolar_zip = pd.read_parquet(PROC / 'deepsolar_zip.parquet')

# Rename year column if needed
if 'install_year' in dgstats_panel.columns:
    dgstats_panel = dgstats_panel.rename(columns={'install_year': 'year'})

panel = build_panel(dgstats_panel, caiso_monthly, deepsolar_zip, zip_utility)
print(f'Panel: {len(panel):,} rows | {panel["zip_code"].nunique():,} ZIPs × {panel["year"].nunique()} years × 12 months')
panel.head(3)

## 4.3 Balance Validation

In [ ]:
expected_combos = panel['zip_code'].nunique() * panel['year'].nunique() * 12
actual = len(panel)
print(f'Expected (balanced): {expected_combos:,} | Actual: {actual:,}')
if actual < expected_combos:
    missing = expected_combos - actual
    print(f'Missing {missing:,} rows — check for gaps in DGStats coverage')
else:
    print('Panel is balanced.')

In [ ]:
panel.to_parquet(PROC / 'panel_for_regression.parquet', index=False)
print('Saved panel_for_regression.parquet')